# Procesos Gaussianos — Regresor (GPR)

**Repositorio:** `autodidact/both/gaussian_processes/`

---

## 1. Idea General e Intuición

Un **Proceso Gaussiano (GP)** es una distribución de probabilidad sobre **funciones**. En lugar de aprender un único conjunto de parámetros, el GP mantiene una distribución completa sobre todas las funciones posibles que podrían explicar los datos.

**Intuición:** Imagina que en lugar de ajustar una recta (un conjunto de parámetros fijo), colocas una "nube de funciones posibles" sobre tus datos. Antes de ver datos (prior), esa nube es amplia. Después de ver datos (posterior), la nube se estrecha alrededor de las observaciones — y la anchura restante representa tu **incertidumbre**.

**Propiedades clave:**
- **Modelo no paramétrico:** La complejidad crece con los datos, no hay número fijo de parámetros.
- **Incertidumbre cuantificada:** Produce intervalos de confianza exactos (gaussianos) en cada punto.
- **Prior sobre funciones:** El kernel define qué tipo de funciones son a priori más probables (suaves, periódicas, etc.).

## 2. Fundamento Matemático

### 2.1 Definición Formal

Un Proceso Gaussiano queda completamente especificado por su **función media** $m(\mathbf{x})$ y su **función de covarianza** (kernel) $k(\mathbf{x}, \mathbf{x}')$:

$$f(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}),\ k(\mathbf{x}, \mathbf{x}'))$$

Para cualquier conjunto finito de puntos $\{x_1, \ldots, x_n\}$, las evaluaciones de $f$ siguen una distribución normal multivariante:

$$\mathbf{f} = [f(x_1), \ldots, f(x_n)]^\top \sim \mathcal{N}(\mathbf{m},\ \mathbf{K})$$

Donde $K_{ij} = k(x_i, x_j)$ es la **matriz de covarianza** (o Gram matrix).

### 2.2 Regresión GP Exacta (Ruido Gaussiano)

Observaciones ruidosas: $y_i = f(x_i) + \varepsilon_i$, $\varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)$.

La **distribución posterior** en puntos de test $\mathbf{X}_*$ dados datos $(\mathbf{X}, \mathbf{y})$ es:

$$f_* \mid \mathbf{X}, \mathbf{y}, \mathbf{X}_* \sim \mathcal{N}(\bar{\mathbf{f}}_*, \text{cov}(\mathbf{f}_*))$$

**Media posterior:**
$$\bar{\mathbf{f}}_* = m(\mathbf{X}_*) + \mathbf{K}_{*X}(\mathbf{K}_{XX} + \sigma_n^2 \mathbf{I})^{-1}(\mathbf{y} - m(\mathbf{X}))$$

**Covarianza posterior:**
$$\text{cov}(\mathbf{f}_*) = \mathbf{K}_{**} - \mathbf{K}_{*X}(\mathbf{K}_{XX} + \sigma_n^2 \mathbf{I})^{-1}\mathbf{K}_{X*}$$

Donde:
- $\mathbf{K}_{XX} = k(\mathbf{X}, \mathbf{X})$: covarianza entre puntos de entrenamiento.
- $\mathbf{K}_{*X} = k(\mathbf{X}_*, \mathbf{X})$: covarianza entre test y entrenamiento.
- $\mathbf{K}_{**} = k(\mathbf{X}_*, \mathbf{X}_*)$: covarianza entre puntos de test.
- $\sigma_n^2$: varianza del ruido.

### 2.3 Kernels Comunes

| Kernel | Fórmula | Función aprendida |
|---|---|---|
| **RBF (Squared Exponential)** | $\sigma^2 \exp\!\left(-\frac{\|\mathbf{x}-\mathbf{x}'\|^2}{2\ell^2}\right)$ | Suave, infinitamente diferenciable |
| **Matérn 3/2** | $\sigma^2(1+\frac{\sqrt{3}r}{\ell})\exp(-\frac{\sqrt{3}r}{\ell})$ | Menos suave que RBF |
| **Matérn 5/2** | $\sigma^2(1+\frac{\sqrt{5}r}{\ell}+\frac{5r^2}{3\ell^2})\exp(-\frac{\sqrt{5}r}{\ell})$ | Intermedio |
| **Lineal** | $\sigma^2 \mathbf{x}^\top\mathbf{x}'$ | Función lineal |
| **Periódico** | $\sigma^2 \exp\!\left(-\frac{2\sin^2(\pi\|\mathbf{x}-\mathbf{x}'\|/p)}{\ell^2}\right)$ | Patrones periódicos |

Donde $\ell$ es el **lengthscale** (escala de correlación) y $\sigma^2$ la amplitud (varianza de señal).

### 2.4 Hiperparámetros: Optimización por Máxima Verosimilitud Marginal

Los hiperparámetros del kernel ($\ell$, $\sigma^2$, $\sigma_n^2$) se optimizan maximizando la **log-verosimilitud marginal**:

$$\log p(\mathbf{y} \mid \mathbf{X}, \theta) = -\frac{1}{2}\mathbf{y}^\top \mathbf{K}_y^{-1} \mathbf{y} - \frac{1}{2}\log|\mathbf{K}_y| - \frac{n}{2}\log 2\pi$$

Donde $\mathbf{K}_y = \mathbf{K}_{XX} + \sigma_n^2 \mathbf{I}$.

Los tres términos representan: **ajuste a los datos** - **complejidad del modelo** - **constante normalizadora**.

### 2.5 Complejidad Computacional

| Operación | Complejidad |
|---|---|
| Entrenamiento | $O(n^3)$ (inversión de $\mathbf{K}$) |
| Predicción | $O(n^2)$ por punto de test |
| Memoria | $O(n^2)$ |

Esto limita el GPR exacto a $n \lesssim 10{,}000$ muestras.

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Dataset **pequeño** (≤ 5000 muestras) y necesitas **incertidumbre cuantificada**.
- Optimización Bayesiana (el GP es el *surrogate model* estándar).
- Datos con **estructura espacial o temporal** conocida (elegir kernel adecuado).
- Necesitas **interpolación exacta** con incertidumbre.
- El prior sobre la forma de la función es informativo.

### ❌ NO lo uses cuando:
- Dataset **grande** (O(n³) es inviable para n > 10k sin aproximaciones).
- Las features tienen **alta dimensionalidad** (el kernel RBF se degrada).
- Necesitas **predicciones ultrarrápidas** en producción.
- No tienes conocimiento para elegir un kernel razonable.

## 4. Métricas de Evaluación

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i-\hat{y}_i)^2}$ | Error de la media posterior |
| **MAE** | $\frac{1}{n}\sum\|y_i-\hat{y}_i\|$ | Error absoluto medio |
| **NLL** | $-\log p(\mathbf{y}\mid\mathbf{X})$ | Calidad de la distribución predictiva completa |
| **Cobertura del IC** | % reales dentro del IC 95% | Calibración de la incertidumbre |
| **R²** | $1-SS_{res}/SS_{tot}$ | Proporción de varianza explicada |

**NLL (Negative Log Likelihood predictivo):** evalúa tanto la precisión de la media como la calibración de los intervalos de confianza. Es la métrica más completa para GPs.

## 5. Problemas y Casos Raros

### 5.1 Inestabilidad Numérica — Matriz No Definida Positiva
La matriz $\mathbf{K}_{XX}$ puede ser casi singular por ruido numérico.
**Solución:** `alpha` en `GaussianProcessRegressor` añade $\alpha \mathbf{I}$ al diagonal (jitter).

### 5.2 Elección del Kernel
El kernel es el prior más importante. Un kernel incorrecto puede dar intervalos mal calibrados.
**Solución:** Comparar kernels por log-verosimilitud marginal.

### 5.3 Múltiples Mínimos Locales
La optimización del kernel por MLE tiene múltiples mínimos locales.
**Solución:** `n_restarts_optimizer > 0` en sklearn.

### 5.4 Maldición de la Dimensionalidad
El RBF en alta dimensión hace que todo esté igualmente cerca.
**Solución:** ARD (Automatic Relevance Determination) — un lengthscale $\ell_j$ por dimensión.

### 5.5 Escalabilidad
Para n > 10k, usar aproximaciones: **Sparse GP** (inducing points), **SGPR**, **VGP**.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, ExpSineSquared,
    WhiteKernel, ConstantKernel as C, DotProduct
)
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
rng  = np.random.RandomState(SEED)
print('✅ Librerías cargadas.')

---
## 6. Visualización del Prior y Posterior — 1D

In [ ]:
# =============================================================================
# PRIOR GP: Muestras de la distribución previa (antes de ver datos)
# El kernel define la forma de las funciones a priori posibles
# =============================================================================
x_plot = np.linspace(-5, 5, 300).reshape(-1, 1)

kernels_prior = {
    'RBF (ℓ=1.0)':     RBF(length_scale=1.0),
    'RBF (ℓ=0.3)':     RBF(length_scale=0.3),
    'Matérn ν=1.5':     Matern(length_scale=1.0, nu=1.5),
    'Periódico (p=2)':  ExpSineSquared(length_scale=1.0, periodicity=2.0)
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (name, kernel) in zip(axes.ravel(), kernels_prior.items()):
    gp_prior = GaussianProcessRegressor(kernel=kernel, random_state=SEED)
    # Muestrear 5 funciones del prior
    y_samples = gp_prior.sample_y(x_plot, n_samples=5, random_state=SEED)
    for i in range(5):
        ax.plot(x_plot, y_samples[:, i], alpha=0.6, lw=1.5)
    ax.set_title(f'Prior — {name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.axhline(0, color='black', lw=0.8, linestyle='--', alpha=0.4)

plt.suptitle('Muestras del Prior GP — Distintos Kernels',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# POSTERIOR GP: Actualización bayesiana con datos observados
# Muestra cómo el GP se ajusta exactamente a los datos y cuantifica incertidumbre
# =============================================================================
# Datos de entrenamiento sintéticos
X_obs = np.array([-4, -2.5, -1, 0.5, 2, 3.5]).reshape(-1, 1)
y_obs = np.sin(X_obs.ravel() * 1.5) + rng.randn(6) * 0.2

kernel_post = C(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.05)
gpr = GaussianProcessRegressor(kernel=kernel_post, n_restarts_optimizer=10, random_state=SEED)
gpr.fit(X_obs, y_obs)

y_mean, y_std = gpr.predict(x_plot, return_std=True)

print(f'Kernel optimizado: {gpr.kernel_}')
print(f'Log-verosimilitud marginal: {gpr.log_marginal_likelihood_value_:.4f}')

fig, ax = plt.subplots(figsize=(11, 5))
# Media posterior
ax.plot(x_plot, y_mean, 'steelblue', lw=2.5, label='Media posterior')
# Intervalos de confianza (±1σ, ±2σ)
ax.fill_between(x_plot.ravel(), y_mean-2*y_std, y_mean+2*y_std,
                alpha=0.15, color='steelblue', label='±2σ (95% IC)')
ax.fill_between(x_plot.ravel(), y_mean-y_std, y_mean+y_std,
                alpha=0.3, color='steelblue', label='±1σ (68% IC)')
# Muestras del posterior
y_post_samples = gpr.sample_y(x_plot, n_samples=3, random_state=SEED)
for s in range(3):
    ax.plot(x_plot, y_post_samples[:, s], alpha=0.5, lw=1, linestyle='--', color='steelblue')
# Puntos observados
ax.scatter(X_obs, y_obs, zorder=10, color='crimson', s=80, edgecolors='k', label='Observaciones')
ax.set_xlabel('x', fontsize=12); ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Posterior GP — Media, Intervalos de Confianza y Muestras',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 7. Comparativa de Kernels en el Posterior

In [ ]:
# =============================================================================
# MISMO DATASET — DISTINTOS KERNELS: observar cómo el kernel modela la función
# =============================================================================
X_obs2 = np.linspace(-4, 4, 12).reshape(-1, 1)
y_obs2 = np.sin(X_obs2.ravel() * 1.5) + np.cos(X_obs2.ravel()) + rng.randn(12) * 0.3

kernels_compare = {
    'RBF':          C(1.0) * RBF(1.0) + WhiteKernel(0.1),
    'Matérn 1.5':   C(1.0) * Matern(1.0, nu=1.5) + WhiteKernel(0.1),
    'Matérn 2.5':   C(1.0) * Matern(1.0, nu=2.5) + WhiteKernel(0.1),
    'RQ':           C(1.0) * RationalQuadratic(1.0, 1.0) + WhiteKernel(0.1)
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, kernel) in zip(axes.ravel(), kernels_compare.items()):
    gpr_k = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=SEED)
    gpr_k.fit(X_obs2, y_obs2)
    y_m, y_s = gpr_k.predict(x_plot, return_std=True)
    lml = gpr_k.log_marginal_likelihood_value_

    ax.fill_between(x_plot.ravel(), y_m-2*y_s, y_m+2*y_s, alpha=0.2, color='steelblue')
    ax.plot(x_plot, y_m, 'steelblue', lw=2)
    ax.scatter(X_obs2, y_obs2, zorder=10, color='crimson', s=60, edgecolors='k')
    ax.set_title(f'Kernel: {name}\nLML={lml:.2f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')

plt.suptitle('Posterior GP con Distintos Kernels — Mismo Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. GPR en Dataset Real — Diabetes

In [ ]:
# =============================================================================
# CARGA Y PREPARACIÓN — Diabetes (pequeño, ideal para GP exacto)
# =============================================================================
diabetes = load_diabetes(as_frame=True)
X, y = diabetes.data.values, diabetes.target.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# OBLIGATORIO: escalar features y target para que el kernel opere en escala normalizada
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_tr_sc = scaler_X.fit_transform(X_train)
X_te_sc = scaler_X.transform(X_test)
y_tr_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

print(f'Train: {X_tr_sc.shape} | Test: {X_te_sc.shape}')
print('Features escaladas y target normalizado.')

In [ ]:
# =============================================================================
# MODELO GPR CON KERNEL COMPUESTO
# C * RBF: amplitud × suavidad + WhiteKernel: ruido
# n_restarts_optimizer evita mínimos locales en la optimización del kernel
# =============================================================================
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(X_tr_sc.shape[1]),
                                     length_scale_bounds=(1e-2, 1e2)) + WhiteKernel(0.1)

gpr_diab = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,              # Jitter numérico para estabilidad
    n_restarts_optimizer=5,
    normalize_y=True,        # Normaliza el target internamente
    random_state=SEED
)
gpr_diab.fit(X_tr_sc, y_train)  # Usamos y_train sin escalar (normalize_y=True)

y_pred_m, y_pred_s = gpr_diab.predict(X_te_sc, return_std=True)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_m))
mae  = mean_absolute_error(y_test, y_pred_m)
r2   = r2_score(y_test, y_pred_m)

print(f'R²   : {r2:.4f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')
print(f'\nKernel optimizado: {gpr_diab.kernel_}')
print(f'LML  : {gpr_diab.log_marginal_likelihood_value_:.4f}')

In [ ]:
# =============================================================================
# CALIBRACIÓN DE INCERTIDUMBRE
# Un GP bien calibrado: ~68% de los puntos dentro de ±1σ, ~95% en ±2σ
# =============================================================================
coverage_1sigma = np.mean(np.abs(y_test - y_pred_m) <= y_pred_s)
coverage_2sigma = np.mean(np.abs(y_test - y_pred_m) <= 2 * y_pred_s)

print(f'Cobertura ±1σ (esperada 68%): {coverage_1sigma*100:.1f}%')
print(f'Cobertura ±2σ (esperada 95%): {coverage_2sigma*100:.1f}%')

# Visualización: predicciones ordenadas con barras de incertidumbre
order = np.argsort(y_test)
x_idx = np.arange(len(y_test))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras de incertidumbre
axes[0].errorbar(x_idx, y_pred_m[order], yerr=2*y_pred_s[order],
                 fmt='none', alpha=0.3, color='steelblue', label='±2σ posterior')
axes[0].scatter(x_idx, y_test[order], s=15, color='crimson', zorder=5, label='Real')
axes[0].scatter(x_idx, y_pred_m[order], s=8, color='steelblue', alpha=0.7, label='Media')
axes[0].set_xlabel('Muestras (ordenadas por y real)')
axes[0].set_ylabel('Respuesta al tratamiento')
axes[0].set_title(f'GPR — Predicciones con Incertidumbre\nCobertura 95%: {coverage_2sigma*100:.1f}%',
                  fontsize=12, fontweight='bold')
axes[0].legend()

# Predichos vs reales
axes[1].scatter(y_test, y_pred_m, alpha=0.5, color='steelblue', s=30)
lim = [y_test.min(), y_test.max()]
axes[1].plot(lim, lim, 'r--', lw=2, label='Predicción perfecta')
axes[1].set_xlabel('Valores Reales'); axes[1].set_ylabel('Media Posterior')
axes[1].set_title(f'Predichos vs Reales  ($R^2$={r2:.4f})', fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('GPR — Diabetes Dataset', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# EFECTO DEL LENGTHSCALE — Interpretación ARD
# Con RBF de longitud por feature, podemos ver qué features son más relevantes:
# un lengthscale grande → feature poco relevante (el GP la ignora)
# =============================================================================
learned_ls = gpr_diab.kernel_.k1.k2.length_scale
feat_df = pd.DataFrame({
    'Feature':     diabetes.feature_names,
    'Lengthscale': learned_ls,
    'Relevancia':  1 / learned_ls   # Inversamente proporcional al lengthscale
}).sort_values('Relevancia', ascending=False)

print('Lengthscales aprendidos por feature (ARD):')
print(feat_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue' if v > feat_df['Lengthscale'].median() else 'crimson'
          for v in feat_df['Lengthscale']]
ax.barh(feat_df['Feature'][::-1], feat_df['Relevancia'][::-1],
        color=colors[::-1], alpha=0.8)
ax.set_xlabel('Relevancia (1 / lengthscale)', fontsize=11)
ax.set_title('ARD: Relevancia de Features aprendida por el Kernel',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Resumen y Checklist

| Aspecto | Detalle |
|---|---|
| **Tipo** | Modelo probabilístico no paramétrico |
| **Complejidad** | O(n³) entrenamiento, O(n²) predicción |
| **Escala máxima** | ≤ 10k muestras (exacto) |
| **Salida** | Media posterior + varianza posterior (incertidumbre) |
| **Kernel** | Define el prior sobre funciones — elección crítica |
| **Hiperparámetros** | Optimizados por MLE marginal |

### ✅ Checklist GPR:
- [ ] ¿Escalé las features y el target?
- [ ] ¿Usé `n_restarts_optimizer > 5` para evitar mínimos locales?
- [ ] ¿Verifiqué la calibración del IC (cobertura 68%/95%)?
- [ ] ¿Comparé LML entre kernels para elegir el mejor?
- [ ] ¿El dataset tiene ≤ 10k muestras?

---
*Ver notebook de clasificación: `gp_clasificador.ipynb`*